# DenseNet121 Cutout Ablation: Shortcut-Learning Audit

This experiment compares three masking strengths with exactly the same patient-free
Kaggle train/validation split, full inverse-frequency sampler, CE loss, standard
DenseNet121 classifier, three-stage schedule, and deterministic seed policy.

The arms are:

| Arm | Masking |
| --- | --- |
| `baseline_random_erasing` | one erase, `p=0.10`, area `2%-5%` |
| `aggressive_single_cutout` | one erase, `p=0.80`, area `2%-10%` |
| `aggressive_double_cutout` | two independent erases, each `p=0.80`, area `2%-8%` |

Grad-CAM is post-training only. It is never used to optimize the loss or select a
checkpoint. Each arm exports classification metrics, failed CAM comparisons, and
a machine-readable manifest under a unique UTC run directory.


In [ ]:
import copy
import csv
import hashlib
import json
import os
import random
import shutil
import subprocess
import time
from collections import Counter, defaultdict
from datetime import datetime, timezone
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np
import timm
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torchvision.transforms as transforms
import tqdm
from sklearn.metrics import (
    average_precision_score,
    classification_report,
    cohen_kappa_score,
    precision_recall_fscore_support,
    recall_score,
    roc_auc_score,
)
from torch.utils.data import DataLoader, Dataset, WeightedRandomSampler

try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    print('Not running in Colab; Drive mount skipped.')

try:
    import torch
    print('PyTorch:', torch.__version__)
except Exception as error:
    raise RuntimeError('Install PyTorch before running this notebook.') from error

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

dataset_zip = '/content/drive/MyDrive/Datasets/kaggle_knee_osteoarthritis.zip'
dataset_root = '/content/Datasets/kaggle_knee_osteoarthritis'
if not os.path.isdir(dataset_root):
    if not os.path.isfile(dataset_zip):
        raise FileNotFoundError(
            f'Dataset ZIP not found at {dataset_zip}. Upload or mount the Kaggle '
            'dataset ZIP at this exact Drive path before running the notebook.'
        )
    os.makedirs('/content/Datasets', exist_ok=True)
    print(f'Extracting {dataset_zip} to /content/Datasets ...')
    subprocess.run(['unzip', '-q', '-o', dataset_zip, '-d', '/content/Datasets'], check=True)
if not os.path.isdir(dataset_root):
    raise FileNotFoundError(
        f'Extraction completed but {dataset_root} was not created. Check the ZIP layout.'
    )
print('Dataset ready:', dataset_root)


## Fixed experiment configuration

The experiment deliberately fixes every non-augmentation choice. The existing
Kaggle `test` split is not used for arm comparison; it remains reserved for a
single final evaluation after selecting an arm.


In [ ]:
class TrainingConfig:
    dataset_root = '/content/Datasets/kaggle_knee_osteoarthritis'
    checkpoint_root = '/content/drive/MyDrive/Models/densenet121_checkpoints'
    run_timestamp = datetime.now(timezone.utc).strftime('%Y-%m-%d_%H-%M-%S_%f_UTC')
    run_root = os.path.join(checkpoint_root, f'{run_timestamp}_cutout_ablation')

    model_name = 'densenet121'
    architecture = 'timm_densenet121_linear_gradcam'
    num_classes = 5
    pretrained = True
    image_size = 384
    batch_size = 64
    num_workers = 4
    seed = 42
    use_amp = True
    classifier_dropout = 0.20

    stage1_epochs = 5
    stage2_epochs = 15
    stage3_epochs = 10
    lr_warmup = 3e-4
    lr_coarse_backbone = 3e-5
    lr_coarse_head = 3e-4
    lr_finetune = 1e-5
    weight_decay = 1e-4

    sampler_power = 1.0
    # Audit every validation ROI; this gate is diagnostic only.
    cam_cases_per_grade = None
    min_joint_energy = 0.55
    max_border_energy = 0.25
    max_lower_tibia_energy = 0.25

    arms = {
        'baseline_random_erasing': {
            'erasers': [(0.10, (0.02, 0.05))],
        },
        'aggressive_single_cutout': {
            'erasers': [(0.80, (0.02, 0.10))],
        },
        'aggressive_double_cutout': {
            'erasers': [(0.80, (0.02, 0.08)), (0.80, (0.02, 0.08))],
        },
    }

os.makedirs(TrainingConfig.run_root, exist_ok=True)


## Dataset and transforms

All arms use the same image paths, labels, split, and sampler. CLAHE and square
padding match the current production preprocessing. No center crop is applied.


In [ ]:
class SquarePadOpenCV:
    def __call__(self, image):
        height, width = image.shape[:2]
        side = max(height, width)
        top = (side - height) // 2
        bottom = side - height - top
        left = (side - width) // 2
        right = side - width - left
        return cv2.copyMakeBorder(
            image, top, bottom, left, right,
            cv2.BORDER_CONSTANT, value=(0, 0, 0)
        )


class OpenCVCLAHE:
    def __call__(self, image_rgb):
        lab = cv2.cvtColor(image_rgb, cv2.COLOR_RGB2LAB)
        lightness, channel_a, channel_b = cv2.split(lab)
        clahe = cv2.createCLAHE(clipLimit=1.25, tileGridSize=(8, 8))
        lightness = clahe.apply(lightness)
        return cv2.cvtColor(
            cv2.merge((lightness, channel_a, channel_b)), cv2.COLOR_LAB2RGB
        )


class MultiRandomErasing:
    def __init__(self, erasers):
        self.erasers = [
            transforms.RandomErasing(
                p=probability,
                scale=scale,
                ratio=(0.5, 2.0),
                value=0,
            )
            for probability, scale in erasers
        ]

    def __call__(self, tensor):
        for eraser in self.erasers:
            tensor = eraser(tensor)
        return tensor


def get_transforms(arm_spec):
    normalize = transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225],
    )
    train_transform = transforms.Compose([
        OpenCVCLAHE(),
        SquarePadOpenCV(),
        transforms.ToPILImage(),
        transforms.RandomHorizontalFlip(p=0.50),
        transforms.RandomRotation(5),
        transforms.ColorJitter(brightness=0.08, contrast=0.08),
        transforms.Resize((TrainingConfig.image_size, TrainingConfig.image_size)),
        transforms.ToTensor(),
        MultiRandomErasing(arm_spec['erasers']),
        normalize,
    ])
    val_transform = transforms.Compose([
        OpenCVCLAHE(),
        SquarePadOpenCV(),
        transforms.ToPILImage(),
        transforms.Resize((TrainingConfig.image_size, TrainingConfig.image_size)),
        transforms.ToTensor(),
        normalize,
    ])
    return train_transform, val_transform


def md5(path):
    digest = hashlib.md5()
    with open(path, 'rb') as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b''):
            digest.update(block)
    return digest.hexdigest()


class KaggleDataset(Dataset):
    def __init__(self, root, split, transform=None, paths=None, labels=None):
        self.root = root
        self.transform = transform
        if paths is not None:
            self.image_paths = list(paths)
            self.labels = list(labels)
            return
        split_root = Path(root) / split
        self.image_paths, self.labels = [], []
        for class_dir in sorted(split_root.iterdir(), key=lambda path: int(path.name)):
            if not class_dir.is_dir() or not class_dir.name.isdigit():
                continue
            for path in sorted(class_dir.iterdir()):
                if path.suffix.lower() in {'.png', '.jpg', '.jpeg'}:
                    self.image_paths.append(str(path))
                    self.labels.append(int(class_dir.name))
        hashes = set()
        unique_paths, unique_labels = [], []
        for path, label in zip(self.image_paths, self.labels):
            image_hash = md5(path)
            if image_hash in hashes:
                continue
            hashes.add(image_hash)
            unique_paths.append(path)
            unique_labels.append(label)
        self.image_paths, self.labels = unique_paths, unique_labels

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, index):
        image = cv2.imread(self.image_paths[index])
        if image is None:
            raise IOError(f'Could not read {self.image_paths[index]}')
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        if self.transform:
            image = self.transform(image)
        return image, self.labels[index]


if not os.path.isdir(TrainingConfig.dataset_root):
    raise FileNotFoundError(
        f"Dataset not found: {TrainingConfig.dataset_root}. "
        "Unzip kaggle_knee_osteoarthritis before running."
    )

base_train = KaggleDataset(TrainingConfig.dataset_root, 'train')
base_val = KaggleDataset(TrainingConfig.dataset_root, 'val')
train_paths, train_labels = base_train.image_paths, base_train.labels
val_paths, val_labels = base_val.image_paths, base_val.labels
print('Fixed split:', len(train_paths), 'train /', len(val_paths), 'validation')
print('Train classes:', dict(sorted(Counter(train_labels).items())))
print('Val classes:', dict(sorted(Counter(val_labels).items())))

class_counts = Counter(train_labels)
sample_weights = [1.0 / (class_counts[label] ** TrainingConfig.sampler_power) for label in train_labels]


## Standard DenseNet121 classifier and fixed training loop

The classifier head is standard global-average pooling plus a linear layer. Grad-CAM
is created later from `features.norm5`; it is not part of training.


In [ ]:
class DenseNet121Classifier(nn.Module):
    def __init__(self, pretrained=True):
        super().__init__()
        self.backbone = timm.create_model(
            'densenet121',
            pretrained=pretrained,
            num_classes=TrainingConfig.num_classes,
            drop_rate=TrainingConfig.classifier_dropout,
        )
        self.use_amp = bool(TrainingConfig.use_amp and torch.cuda.is_available())
        self.scaler = torch.amp.GradScaler('cuda', enabled=self.use_amp)

    @property
    def classifier(self):
        return self.backbone.classifier

    @property
    def gradcam_target_layer(self):
        return self.backbone.features.norm5

    def forward(self, images):
        return self.backbone(images)

    def freeze_backbone(self):
        for parameter in self.backbone.parameters():
            parameter.requires_grad = False
        for parameter in self.classifier.parameters():
            parameter.requires_grad = True

    def unfreeze_last_block(self):
        for parameter in self.parameters():
            parameter.requires_grad = False
        for name, module in self.backbone.named_modules():
            if 'denseblock3' in name or 'denseblock4' in name or 'norm5' in name:
                for parameter in module.parameters():
                    parameter.requires_grad = True
        for parameter in self.classifier.parameters():
            parameter.requires_grad = True

    def unfreeze_all(self):
        for parameter in self.parameters():
            parameter.requires_grad = True


def evaluate(model, loader):
    model.eval()
    all_labels, all_predictions, all_probabilities = [], [], []
    total_loss = 0.0
    criterion = nn.CrossEntropyLoss()
    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            # Match the Grad-CAM audit and production scoring precision.
            logits = model(images)
            loss = criterion(logits, labels)
            probabilities = F.softmax(logits.float(), dim=1)
            total_loss += float(loss.item()) * labels.size(0)
            all_labels.extend(labels.cpu().numpy())
            all_predictions.extend(probabilities.argmax(1).cpu().numpy())
            all_probabilities.extend(probabilities.cpu().numpy())
    labels = np.asarray(all_labels)
    predictions = np.asarray(all_predictions)
    probabilities = np.asarray(all_probabilities)
    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, predictions, labels=list(range(5)), average='macro', zero_division=0
    )
    one_hot = np.eye(5)[labels]
    try:
        macro_ap = average_precision_score(one_hot, probabilities, average='macro')
    except ValueError:
        macro_ap = float('nan')
    try:
        macro_auc = roc_auc_score(one_hot, probabilities, average='macro')
    except ValueError:
        macro_auc = float('nan')
    qwk = cohen_kappa_score(labels, predictions, weights='quadratic')
    grade1_recall = recall_score(labels == 1, predictions == 1, zero_division=0)
    return {
        'loss': total_loss / max(1, len(labels)),
        'accuracy': float(np.mean(labels == predictions)),
        'qwk': float(qwk),
        'macro_precision': float(precision),
        'macro_recall': float(recall),
        'macro_f1': float(f1),
        'grade1_recall': float(grade1_recall),
        'macro_ap': float(macro_ap),
        'macro_auc': float(macro_auc),
        'labels': labels,
        'predictions': predictions,
        'probabilities': probabilities,
        'classification_report': classification_report(
            labels, predictions, labels=list(range(5)), zero_division=0
        ),
    }


def selection_score(metrics):
    return 0.55 * metrics['qwk'] + 0.30 * metrics['macro_f1'] + 0.15 * metrics['macro_ap']


def build_loaders(arm_name, arm_spec):
    train_transform, val_transform = get_transforms(arm_spec)
    train_data = KaggleDataset(
        TrainingConfig.dataset_root, 'train', transform=train_transform,
        paths=train_paths, labels=train_labels,
    )
    val_data = KaggleDataset(
        TrainingConfig.dataset_root, 'val', transform=val_transform,
        paths=val_paths, labels=val_labels,
    )
    sampler = WeightedRandomSampler(
        sample_weights,
        num_samples=len(sample_weights),
        replacement=True,
        generator=torch.Generator().manual_seed(TrainingConfig.seed),
    )
    loader_kwargs = {
        'num_workers': TrainingConfig.num_workers,
        'pin_memory': torch.cuda.is_available(),
        'persistent_workers': TrainingConfig.num_workers > 0,
    }
    train_loader = DataLoader(
        train_data, batch_size=TrainingConfig.batch_size, sampler=sampler,
        generator=torch.Generator().manual_seed(TrainingConfig.seed), **loader_kwargs
    )
    val_loader = DataLoader(
        val_data, batch_size=TrainingConfig.batch_size, shuffle=False, **loader_kwargs
    )
    return train_data, val_data, train_loader, val_loader


def train_one_epoch(model, loader, optimizer):
    model.train()
    criterion = nn.CrossEntropyLoss()
    total_loss, total, correct = 0.0, 0, 0
    for images, labels in tqdm.tqdm(loader, leave=False):
        images, labels = images.to(device, non_blocking=True), labels.to(device, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        with torch.amp.autocast(device_type=device.type, enabled=model.use_amp):
            logits = model(images)
            loss = criterion(logits, labels)
        model.scaler.scale(loss).backward()
        model.scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        model.scaler.step(optimizer)
        model.scaler.update()
        total_loss += float(loss.item()) * labels.size(0)
        total += labels.size(0)
        correct += int((logits.argmax(1) == labels).sum().item())
    return total_loss / max(1, total), correct / max(1, total)


def save_checkpoint(path, model, optimizer, scheduler, arm_name, epoch, metrics, history):
    checkpoint = {
        'epoch': epoch,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'scheduler_state_dict': scheduler.state_dict() if scheduler else None,
        'model_name': TrainingConfig.model_name,
        'architecture': TrainingConfig.architecture,
        'loss_type': 'ce',
        'run_timestamp': TrainingConfig.run_timestamp,
        'arm': arm_name,
        'validation_metrics': {key: value for key, value in metrics.items() if isinstance(value, (int, float))},
        'training_config': {
            'batch_size': TrainingConfig.batch_size,
            'image_size': TrainingConfig.image_size,
            'sampler_power': TrainingConfig.sampler_power,
            'stage_schedule': [TrainingConfig.stage1_epochs, TrainingConfig.stage2_epochs, TrainingConfig.stage3_epochs],
            'arm': TrainingConfig.arms[arm_name],
        },
        'history': history,
    }
    torch.save(checkpoint, path)


def train_arm(arm_name, arm_spec):
    print(f'\n===== TRAINING {arm_name} =====')
    random.seed(TrainingConfig.seed)
    np.random.seed(TrainingConfig.seed)
    torch.manual_seed(TrainingConfig.seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(TrainingConfig.seed)
    train_data, val_data, train_loader, val_loader = build_loaders(arm_name, arm_spec)
    arm_dir = Path(TrainingConfig.run_root) / arm_name
    arm_dir.mkdir(parents=True, exist_ok=True)
    model = DenseNet121Classifier(pretrained=TrainingConfig.pretrained).to(device)
    history = []
    stage2_path = arm_dir / 'stage2_best_model.pth'
    best_path = arm_dir / 'best_model.pth'
    best_stage2_score = -float('inf')
    best_stage3_score = -float('inf')
    best_epoch = None
    current_stage = None
    optimizer = scheduler = None
    total_epochs = TrainingConfig.stage1_epochs + TrainingConfig.stage2_epochs + TrainingConfig.stage3_epochs

    for epoch in range(total_epochs):
        if epoch < TrainingConfig.stage1_epochs:
            stage = 'stage1'
            if current_stage != stage:
                model.freeze_backbone()
                optimizer = optim.AdamW(model.classifier.parameters(), lr=TrainingConfig.lr_warmup, weight_decay=TrainingConfig.weight_decay)
                scheduler = None
                current_stage = stage
        elif epoch < TrainingConfig.stage1_epochs + TrainingConfig.stage2_epochs:
            stage = 'stage2'
            if current_stage != stage:
                model.unfreeze_last_block()
                head_ids = {id(parameter) for parameter in model.classifier.parameters()}
                backbone_parameters = [
                    parameter for parameter in model.backbone.parameters()
                    if parameter.requires_grad and id(parameter) not in head_ids
                ]
                optimizer = optim.AdamW([
                    {'params': backbone_parameters, 'lr': TrainingConfig.lr_coarse_backbone},
                    {'params': model.classifier.parameters(), 'lr': TrainingConfig.lr_coarse_head},
                ], weight_decay=TrainingConfig.weight_decay)
                scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=TrainingConfig.stage2_epochs, eta_min=1e-7)
                current_stage = stage
        else:
            stage = 'stage3'
            if current_stage != stage:
                if stage2_path.exists():
                    checkpoint = torch.load(stage2_path, map_location=device, weights_only=False)
                    model.load_state_dict(checkpoint['model_state_dict'])
                model.unfreeze_all()
                optimizer = optim.AdamW(model.parameters(), lr=TrainingConfig.lr_finetune, weight_decay=10 * TrainingConfig.weight_decay)
                scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=TrainingConfig.stage3_epochs, eta_min=1e-7)
                current_stage = stage

        train_loss, train_accuracy = train_one_epoch(model, train_loader, optimizer)
        metrics = evaluate(model, val_loader)
        score = selection_score(metrics)
        history.append({
            'epoch': epoch + 1,
            'stage': stage,
            'train_loss': train_loss,
            'train_accuracy': train_accuracy,
            'selection_score': score,
            **{key: value for key, value in metrics.items() if isinstance(value, (int, float))},
        })
        if scheduler is not None:
            scheduler.step()
        print(
            f'{arm_name} | epoch {epoch + 1}/{total_epochs} | {stage} | '
            f'QWK={metrics["qwk"]:.4f} | F1={metrics["macro_f1"]:.4f} | '
            f'AP={metrics["macro_ap"]:.4f} | selection={score:.4f}'
        )
        if stage == 'stage2' and score > best_stage2_score:
            best_stage2_score = score
            torch.save({
                'model_state_dict': model.state_dict(),
                'validation_metrics': {key: value for key, value in metrics.items() if isinstance(value, (int, float))},
                'epoch': epoch + 1,
            }, stage2_path)
        if stage == 'stage3' and score > best_stage3_score:
            best_stage3_score = score
            best_epoch = epoch + 1
            save_checkpoint(best_path, model, optimizer, scheduler, arm_name, epoch + 1, metrics, history)

    if not best_path.exists():
        save_checkpoint(best_path, model, optimizer, scheduler, arm_name, total_epochs, metrics, history)
        best_epoch = total_epochs
    checkpoint_hash = hashlib.sha256(best_path.read_bytes()).hexdigest()
    manifest = {
        'arm': arm_name,
        'run_timestamp': TrainingConfig.run_timestamp,
        'checkpoint': str(best_path),
        'checkpoint_sha256': checkpoint_hash,
        'best_epoch': best_epoch,
        'architecture': TrainingConfig.architecture,
        'loss': 'ce',
        'sampler': 'full_inverse_frequency',
        'arm_config': arm_spec,
        'history': history,
    }
    (arm_dir / 'run_manifest.json').write_text(json.dumps(manifest, indent=2, default=float))
    return {
        'arm': arm_name,
        'checkpoint': str(best_path),
        'arm_dir': str(arm_dir),
        'best_epoch': best_epoch,
        'best_selection_score': best_stage3_score,
        'history': history,
    }


## Run all three arms

This is the long-running cell. Let it finish; each arm writes a checkpoint and
manifest immediately, so completed arms are preserved if a Colab runtime stops.


In [ ]:
arm_results = {}
for arm_name, arm_spec in TrainingConfig.arms.items():
    arm_results[arm_name] = train_arm(arm_name, arm_spec)

with open(os.path.join(TrainingConfig.run_root, 'arm_results.json'), 'w') as handle:
    json.dump(arm_results, handle, indent=2, default=float)
print(json.dumps({
    name: {
        'best_epoch': result['best_epoch'],
        'best_selection_score': result['best_selection_score'],
        'checkpoint': result['checkpoint'],
    }
    for name, result in arm_results.items()
}, indent=2))


## Grad-CAM audit: same validation cases for every arm

The audit uses up to 50 cases per true grade, identical cases for all arms. It
exports original ROI, exact processed input, and predicted-class Grad-CAM for
every failed anatomy gate. These maps are diagnostic only.


In [ ]:
class GradCAM:
    def __init__(self, model):
        self.model = model
        self.activations = None
        self.gradients = None
        self.handle = model.gradcam_target_layer.register_forward_hook(self.capture)

    def capture(self, module, inputs, output):
        self.activations = output

    def capture_gradient(self, gradient):
        self.gradients = gradient

    def remove(self):
        self.handle.remove()

    def __call__(self, tensor):
        self.model.eval()
        self.model.zero_grad(set_to_none=True)
        self.activations = None
        self.gradients = None
        with torch.enable_grad():
            logits = self.model(tensor.detach().clone().requires_grad_(True))
            predicted = int(logits.argmax(1).item())
            if self.activations is None or not self.activations.requires_grad:
                raise RuntimeError('Could not capture Grad-CAM activation.')
            self.activations.register_hook(self.capture_gradient)
            logits[0, predicted].backward()
        weights = self.gradients.mean(dim=(2, 3), keepdim=True)
        cam = F.relu((weights * self.activations).sum(dim=1, keepdim=True))
        cam = F.interpolate(cam, size=tensor.shape[-2:], mode='bilinear', align_corners=False)[0, 0]
        cam = cam.detach().cpu().numpy()
        cam /= max(float(cam.max()), 1e-8)
        return logits.detach(), predicted, cam


def prepare_cam_input(path):
    image_bgr = cv2.imread(path)
    image_rgb = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)
    processed_rgb = np.asarray(transforms.Compose([
        OpenCVCLAHE(), SquarePadOpenCV(), transforms.ToPILImage(),
        transforms.Resize((TrainingConfig.image_size, TrainingConfig.image_size)),
    ])(image_rgb))
    tensor = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
    ])(processed_rgb).unsqueeze(0).to(device)
    return image_rgb, processed_rgb, tensor


def cam_metrics(cam):
    height, width = cam.shape
    joint = np.zeros_like(cam, dtype=bool)
    joint[int(.28 * height):int(.72 * height), int(.06 * width):int(.94 * width)] = True
    border = np.ones_like(cam, dtype=bool)
    border[int(.08 * height):int(.92 * height), int(.08 * width):int(.92 * width)] = False
    lower = np.zeros_like(cam, dtype=bool)
    lower[int(.72 * height):, :] = True
    total = float(cam.sum()) + 1e-8
    peak = np.unravel_index(np.argmax(cam), cam.shape)
    return {
        'joint_energy': float(cam[joint].sum() / total),
        'border_energy': float(cam[border].sum() / total),
        'lower_tibia_energy': float(cam[lower].sum() / total),
        'peak_inside_joint': bool(joint[peak]),
    }


def cam_failures(metrics):
    reasons = []
    if metrics['joint_energy'] < TrainingConfig.min_joint_energy:
        reasons.append('low_joint_energy')
    if metrics['border_energy'] > TrainingConfig.max_border_energy:
        reasons.append('high_border_energy')
    if metrics['lower_tibia_energy'] > TrainingConfig.max_lower_tibia_energy:
        reasons.append('high_lower_tibia_energy')
    if not metrics['peak_inside_joint']:
        reasons.append('peak_outside_joint')
    return reasons


def save_cam_comparison(original, processed, cam, row, path):
    heatmap = cv2.applyColorMap(np.uint8(np.clip(cam, 0, 1) * 255), cv2.COLORMAP_JET)
    heatmap = cv2.cvtColor(heatmap, cv2.COLOR_BGR2RGB)
    overlay = cv2.addWeighted(processed, .60, heatmap, .40, 0)
    figure, axes = plt.subplots(1, 3, figsize=(15, 5))
    axes[0].imshow(original)
    axes[0].set_title(f'Original ROI | true G{row["true_grade"]}')
    axes[1].imshow(processed)
    axes[1].set_title('CLAHE 1.25 -> pad -> 384')
    axes[2].imshow(overlay)
    axes[2].set_title(f'Grad-CAM predicted G{row["predicted_grade"]}')
    for axis in axes:
        axis.axis('off')
    figure.suptitle(
        f'joint={row["joint_energy"]:.3f} border={row["border_energy"]:.3f} '
        f'lower={row["lower_tibia_energy"]:.3f} | {row["failure_reasons"] or "pass"}',
        fontsize=10,
    )
    figure.tight_layout()
    figure.savefig(path, dpi=150, bbox_inches='tight')
    plt.close(figure)


audit_indices = []
for grade in range(TrainingConfig.num_classes):
    candidates = [index for index, label in enumerate(val_labels) if label == grade]
    limit = TrainingConfig.cam_cases_per_grade
    audit_indices.extend(candidates if limit is None else candidates[:limit])
audit_indices = sorted(set(audit_indices))
print('Shared CAM audit cases:', len(audit_indices))

audit_summary = {}
for arm_name, arm_result in arm_results.items():
    arm_dir = Path(arm_result['arm_dir'])
    audit_dir = arm_dir / 'gradcam_audit'
    failed_dir = audit_dir / 'failed'
    audit_dir.mkdir(exist_ok=True)
    failed_dir.mkdir(exist_ok=True)
    train_transform, val_transform = get_transforms(TrainingConfig.arms[arm_name])
    model = DenseNet121Classifier(pretrained=False).to(device)
    checkpoint = torch.load(arm_result['checkpoint'], map_location=device, weights_only=False)
    model.load_state_dict(checkpoint['model_state_dict'])
    gradcam = GradCAM(model)
    rows = []
    failed_paths = []
    for audit_number, index in enumerate(tqdm.tqdm(audit_indices, desc=f'CAM {arm_name}')):
        original, processed, tensor = prepare_cam_input(val_paths[index])
        logits, predicted, cam = gradcam(tensor)
        probabilities = F.softmax(logits.float(), dim=1)[0]
        metrics = cam_metrics(cam)
        reasons = cam_failures(metrics)
        row = {
            'dataset_index': index,
            'filename': os.path.basename(val_paths[index]),
            'true_grade': int(val_labels[index]),
            'predicted_grade': predicted,
            'confidence': float(probabilities[predicted].item()),
            'correct': int(predicted == int(val_labels[index])),
            'failure_reasons': ';'.join(reasons),
            **metrics,
        }
        rows.append(row)
        if reasons:
            figure_path = failed_dir / f'{audit_number:04d}_true_G{row["true_grade"]}_pred_G{predicted}.jpg'
            save_cam_comparison(original, processed, cam, row, figure_path)
            failed_paths.append(str(figure_path))
    gradcam.remove()
    with open(audit_dir / 'all_cases.csv', 'w', newline='') as handle:
        writer = csv.DictWriter(handle, fieldnames=list(rows[0]))
        writer.writeheader()
        writer.writerows(rows)
    failed_rows = [row for row in rows if row['failure_reasons']]
    with open(audit_dir / 'failed_cases.csv', 'w', newline='') as handle:
        writer = csv.DictWriter(handle, fieldnames=list(rows[0]))
        writer.writeheader()
        writer.writerows(failed_rows)
    summary = {
        'arm': arm_name,
        'cases': len(rows),
        'gate_passes': len(rows) - len(failed_rows),
        'gate_failures': len(failed_rows),
        'gate_failure_rate': len(failed_rows) / max(1, len(rows)),
        'failure_reasons': dict(Counter(reason for row in failed_rows for reason in row['failure_reasons'].split(';') if reason)),
        'failed_by_true_grade': dict(Counter(row['true_grade'] for row in failed_rows)),
        'thresholds': {
            'minimum_joint_energy': TrainingConfig.min_joint_energy,
            'maximum_border_energy': TrainingConfig.max_border_energy,
            'maximum_lower_tibia_energy': TrainingConfig.max_lower_tibia_energy,
        },
    }
    (audit_dir / 'summary.json').write_text(json.dumps(summary, indent=2))
    audit_summary[arm_name] = summary

with open(os.path.join(TrainingConfig.run_root, 'gradcam_comparison_summary.json'), 'w') as handle:
    json.dump(audit_summary, handle, indent=2)
print(json.dumps(audit_summary, indent=2))


## Arm comparison and report

The winner is selected by validation QWK + macro F1 + macro AP. CAM failure rates
are reported alongside the metrics and must be reviewed before production use.


In [ ]:
comparison = []
for arm_name, result in arm_results.items():
    history = result['history']
    best = next(
        row for row in history
        if row['epoch'] == result['best_epoch'] and row['stage'] == 'stage3'
    )
    comparison.append({
        'arm': arm_name,
        'best_epoch': result['best_epoch'],
        'selection_score': result['best_selection_score'],
        'qwk': best['qwk'],
        'macro_f1': best['macro_f1'],
        'macro_ap': best['macro_ap'],
        'grade1_recall': best['grade1_recall'],
        'cam_failure_rate': audit_summary[arm_name]['gate_failure_rate'],
        'checkpoint': result['checkpoint'],
    })
comparison.sort(key=lambda row: row['selection_score'], reverse=True)
with open(os.path.join(TrainingConfig.run_root, 'arm_comparison.csv'), 'w', newline='') as handle:
    writer = csv.DictWriter(handle, fieldnames=list(comparison[0]))
    writer.writeheader()
    writer.writerows(comparison)
report = [
    f'# DenseNet121 Cutout Ablation - {TrainingConfig.run_timestamp}',
    '',
    '| Arm | Best epoch | QWK | Macro F1 | Macro AP | Grade 1 recall | CAM failure rate |',
    '| --- | ---: | ---: | ---: | ---: | ---: | ---: |',
]
for row in comparison:
    report.append(
        f"| {row['arm']} | {row['best_epoch']} | {row['qwk']:.4f} | "
        f"{row['macro_f1']:.4f} | {row['macro_ap']:.4f} | "
        f"{row['grade1_recall']:.4f} | {row['cam_failure_rate']:.2%} |"
    )
report.extend([
    '',
    f"Metric winner: `{comparison[0]['arm']}`.",
    '',
    'CAM failure rate is a diagnostic, not a training objective. Do not promote an arm based on classification metrics alone if its failed CAM gallery shows border or padding shortcut attention.',
])
(Path(TrainingConfig.run_root) / 'report.md').write_text('\n'.join(report) + '\n')
print('\n'.join(report))
